# Prepare required libraries

In [1]:
# import

## 💻 Data Processing and Core Utilities
import os
import shutil
import uuid
import json
import pickle
import copy
import warnings
import pprint
import datetime as dt

import pandas as pd
import numpy as np
from numpy import array, random, arange, typing as npt
from functools import reduce

from collections import Counter
from typing import Dict, List, Tuple, Union, Literal
from scipy import stats

## 🤖 Machine Learning Models and Preprocessing
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures, FunctionTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

## 📊 Model Evaluation and Tuning
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.metrics import (
    roc_auc_score,
    fbeta_score,
    make_scorer,
    precision_score,
    accuracy_score,
    confusion_matrix,
    auc,
)
from sklearn.feature_selection import (
    RFE,
    SequentialFeatureSelector as SFS,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    f_classif,
)
import optuna

## 📈 Visualization and Other Libraries
import matplotlib.pyplot as plt
import plotly.express as px
import shap


In [2]:
# Module for handling warning messages

# Ignore only the 'use_label_encoder' warning.
warnings.filterwarnings("ignore")

In [3]:
# Define the required classes

# Custom class for JSON serialization
class NpEncoder(json.JSONEncoder):
    """Class for serializing Numpy types (int64, float64, etc.) to JSON"""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)
    
# Defines a class named 'BaselineModel'. This model is a simple rule-based baseline model.
class BaselineModel:
    def __init__(self):
        # Initializes the baseline values.
        self.radius = 70  # Radius threshold
        self.sensor_offset_hot_cold = 0.02 # Sensor offset threshold
        pass

    # Function that returns prediction probabilities (similar to a machine learning model's predict_proba)
    def predict_proba(self, X):
        # Create boolean (True/False) series for each criterion.
        radius_criteria = X["Radius"] <= self.radius # True if radius is less than or equal to the threshold
        sensor_criteria = X["SensorOffsetHot-Cold"].abs() <= self.sensor_offset_hot_cold # True if the absolute sensor offset is less than or equal to the threshold
        bandgap_criteria = X["band gap dpat_ok for band gap"] == 1 # Checks if the bandgap criterion is met in the test data
        # Determine the final prediction by checking if all criteria are met.
        y_pred_baseline = radius_criteria & sensor_criteria & bandgap_criteria

        # proba[:,1] => "pass", proba[:,0] => "fail"
        # Create a numpy array with 2 columns to store prediction probabilities.
        proba = np.zeros((len(X), 2))
        # Store 1 for 'pass' (True) and 0 for 'fail' (False) in the 'pass' column.
        proba[:, 0] = y_pred_baseline.astype(int)  # 'Pass' probability (actually 0 or 1)
        # Store the inverse of the 'pass' probability in the 'fail' column, since 'fail' is the opposite of 'pass'.
        proba[:, 1] = 1 - proba[:, 0]  # 'Fail' probability
        return proba

In [4]:
# Define the necessary functions

def rescale_df(df: pd.DataFrame, scaler_type: str = 'standard') -> pd.DataFrame:
    """
    Rescales numeric variables in a DataFrame using a specified scaler.
    Non-numeric variables are kept as is.

    Args:
        df (pd.DataFrame): The input DataFrame to rescale.
        scaler_type (str): The type of scaler to use ('standard', 'minmax', 'robust').

    Returns:
        pd.DataFrame: The rescaled DataFrame (only numeric columns are transformed).
    
    # --- Example Usage ---

    # Create a DataFrame with various data types
    data = {
        'numerical_feature_1': [10, 20, 30, 40, 50],
        'numerical_feature_2': [100, 200, 300, 400, 500],
        'categorical_feature': ['A', 'B', 'A', 'C', 'B'],
        'object_feature': ['apple', 'banana', 'orange', 'grape', 'apple']
    }
    df = pd.DataFrame(data)
    print("Original DataFrame:")
    print(df)
    print("\n" + "="*30 + "\n")

    # Rescale using MinMaxScaler
    df_minmax_rescaled = rescale_df(df, scaler_type='minmax')
    print("Rescaled DataFrame (using MinMaxScaler):")
    print(df_minmax_rescaled)
    print("\n" + "="*30 + "\n")

    # Rescale using RobustScaler
    df_robust_rescaled = rescale_df(df, scaler_type='robust')
    print("Rescaled DataFrame (using RobustScaler):")
    print(df_robust_rescaled)
    print("\n" + "="*30 + "\n")

    # Input with an unsupported scaler type
    df_error = rescale_df(df, scaler_type='unsupported_scaler')    
        
    """
    # Select scaler object based on type
    if scaler_type == 'standard':
        scaler = StandardScaler()
    elif scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif scaler_type == 'robust':
        scaler = RobustScaler()
    else:
        print(f"Error: Unsupported scaler type '{scaler_type}'. Please choose from 'standard', 'minmax', 'robust'.")
        return df

    # Create a copy of the original DataFrame to avoid modifying it
    df_rescaled = df.copy()

    # Select only numeric (int, float) columns
    numeric_cols = df_rescaled.select_dtypes(include=np.number).columns
    
    # Check if there are any columns to rescale
    if numeric_cols.empty:
        print("Warning: No numeric columns found for scaling.")
        return df_rescaled

    try:
        # Apply fit_transform to the selected numeric columns
        df_rescaled[numeric_cols] = scaler.fit_transform(df_rescaled[numeric_cols])

    except ValueError as e:
        print(f"Error during scaling: {e}")
        print("Check for missing values (NaN) or infinite values in your data.")
        return df

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return df

    return df_rescaled

# Function to calculate TPR (True Positive Rate) and FPR (False Positive Rate)
def true_false_positive(threshold_vector: np.array, y_test: np.array):
    # "1" is 'fail', which is positive.
    true_positive = (threshold_vector == 1) & (y_test == 1) # TP: pred=1 & actual=1
    false_positive = (threshold_vector == 1) & (y_test == 0) # FP: pred=1 & actual=0
    true_negative = (threshold_vector == 0) & (y_test == 0) # TN: pred=0 & actual=0
    false_negative = (threshold_vector == 0) & (y_test == 1) # FN: pred=0 & actual=1

    # Calculate TPR: TP / (TP + FN)
    tpr = true_positive.sum() / (true_positive.sum() + false_negative.sum() + 1e-9)
    # Calculate FPR: FP / (FP + TN)
    fpr = false_positive.sum() / (false_positive.sum() + true_negative.sum() + 1e-9)
    return tpr, fpr


# Function to generate labels for the Confusion Matrix
def _confusion_label(row):
    # Now, "1" is 'fail', which is considered positive.
    # row["Historical"] = actual label, row["Forecast"] = predicted label
    if row["Historical"] == 1 and row["Forecast"] == 1:
        return "True Fail (TP)" # Correctly predicted a fail as a fail
    elif row["Historical"] == 0 and row["Forecast"] == 0:
        return "True Pass (TN)" # Correctly predicted a pass as a pass
    elif row["Historical"] == 0 and row["Forecast"] == 1:
        return "False Fail (FP)" # Incorrectly predicted a pass as a fail (error)
    else:  # row["Historical"] == 1 and row["Forecast"] == 0
        return "Missed Fail (FN)" # Incorrectly predicted a fail as a pass (missed)


In [5]:
# Function to create training and test datasets
def create_train_test_data(
    preprocessed_dataset: pd.DataFrame,
    split_parameter: dict = None
):
    """
    Function that applies training/test data splitting and sampling.
    Feature Generation application and options are controlled by split_parameter.
    Detailed processing results are added to split_parameter_info.
    """
    print("\n\n##############################################################################################################################")
    print("# 3) Create Train/Test Split ")
    print("##############################################################################################################################")
    
    print("\n      Creating training and test datasets...")

    # Set and update default values for split_parameter
    default_params = {
        'test_size': 0.2,
        'random_state': 42,
        'sampling_ratio': None,
        'apply_feature_generation': False,
        'sum_features': False,
        'diff_features': False,
        'poly_features': False,
        'poly_degree': 2,
        'apply_filter_split': False,
        'var_threshold_split': 0.0,
        'corr_threshold_split': 0.98,
        'apply_filter_gen': False,
        'var_threshold_gen': 0.0,
        'corr_threshold_gen': 0.98,
    }
    
    if split_parameter:
        default_params.update(split_parameter)
    split_parameter = default_params

    split_parameter_info = split_parameter.copy()
    
    # ----------------------------------------------------
    # Step 1: Remove outliers
    # ----------------------------------------------------
    initial_dataset_shape = preprocessed_dataset.shape
    outlier_mask = (preprocessed_dataset["Radius"] < 32) & (preprocessed_dataset["Pass/Fail"])
    preprocessed_dataset = preprocessed_dataset[~outlier_mask].reset_index(drop=True)
    split_parameter_info['rows_after_outlier_removal'] = preprocessed_dataset.shape[0]

    X = preprocessed_dataset.iloc[:, :-1]
    y = preprocessed_dataset.iloc[:, -1]
    
    # ----------------------------------------------------
    # Step 2: Apply filtering before splitting
    # ----------------------------------------------------
    split_parameter_info['features_before_split_filter'] = X.shape[1]
    if split_parameter['apply_filter_split']:
        print(f"    - Applying filtering before splitting (Variance: {split_parameter['var_threshold_split']}, Correlation: {split_parameter['corr_threshold_split']})")
        # X, _, var_dropped, corr_dropped = variance_correlation_filter(X, var_threshold=split_parameter['var_threshold_split'], corr_threshold=split_parameter['corr_threshold_split'])
        X, _, var_dropped, corr_dropped = filter_by_variance(X, 0)
        split_parameter_info['features_after_split_filter'] = X.shape[1]
        split_parameter_info['features_dropped_by_variance_split'] = var_dropped
        split_parameter_info['features_dropped_by_correlation_split'] = corr_dropped
    else:
        print("    - Not applying filtering before splitting.")
        split_parameter_info['features_after_split_filter'] = X.shape[1]
        split_parameter_info['features_dropped_by_variance_split'] = 0
        split_parameter_info['features_dropped_by_correlation_split'] = 0
        
    # ----------------------------------------------------
    # Step 3: Apply Feature Generation
    # ----------------------------------------------------
    split_parameter_info['original_feature_count'] = X.shape[1]
    if split_parameter['apply_feature_generation']:
        print("    - Applying Feature Generation...")
        X, gen_counts = feature_generator(
            X,  
            sum_features=split_parameter['sum_features'],
            diff_features=split_parameter['diff_features'],
            poly_features=split_parameter['poly_features'],
            poly_degree=split_parameter['poly_degree'],
            apply_filter_gen=split_parameter['apply_filter_gen'],
            var_threshold_gen=split_parameter['var_threshold_gen'],
            corr_threshold_gen=split_parameter['corr_threshold_gen']
        )
        split_parameter_info['generated_feature_counts'] = gen_counts
        split_parameter_info['total_generated_features'] = sum(gen_counts.values())
        split_parameter_info['features_after_generation'] = X.shape[1]
        
        generation_types = []
        if split_parameter['sum_features']: generation_types.append('sum')
        if split_parameter['diff_features']: generation_types.append('diff')
        if split_parameter['poly_features']: generation_types.append('poly')
        split_parameter_info['generation_types_applied'] = generation_types
        
        print("    - Feature Generation complete. New feature count:", split_parameter_info['total_generated_features'])
    else:
        print("    - Not applying Feature Generation.")
        split_parameter_info['generated_feature_counts'] = {'sum': 0, 'diff': 0, 'poly': 0}
        split_parameter_info['total_generated_features'] = 0
        split_parameter_info['features_after_generation'] = X.shape[1]

    # ----------------------------------------------------
    # Step 4: Split data into training/test sets
    # ----------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,  
        test_size=split_parameter['test_size'],  
        random_state=split_parameter['random_state'],  
        stratify=y
    )
    
    split_parameter_info['train_samples_before_sampling'] = len(X_train)
    split_parameter_info['test_samples'] = len(X_test)
    
    train_class_distribution_before = dict(sorted(Counter(y_train).items()))
    split_parameter_info['class_distribution_before_sampling'] = train_class_distribution_before
    print(f"\n    - Training data class distribution before splitting: {train_class_distribution_before}")

    # ----------------------------------------------------
    # Step 5: Apply sampling
    # ----------------------------------------------------
    sampling_ratio = split_parameter['sampling_ratio']
    
    if sampling_ratio is not None:
        split_parameter_info['sampling_ratio_used'] = sampling_ratio
        if sampling_ratio >= 1:
            n_samples_majority = sum(y_train == 0)
            target_minority_count = int(n_samples_majority * sampling_ratio)
            sampling_strategy = {1: target_minority_count}
            sampler = SMOTE(sampling_strategy=sampling_strategy, random_state=split_parameter['random_state'])
            X_train, y_train = sampler.fit_resample(X_train, y_train)
            print(f"    - Oversampling applied (minority class ratio: {sampling_ratio})")
            split_parameter_info['sampling_applied'] = 'oversampling'
        else:
            n_samples_minority = sum(y_train == 1)
            target_majority_count = int(n_samples_minority / sampling_ratio)
            sampling_strategy = {0: target_majority_count}
            sampler = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=split_parameter['random_state'])
            X_train, y_train = sampler.fit_resample(X_train, y_train)
            print(f"    - Undersampling applied (minority class ratio: {sampling_ratio})")
            split_parameter_info['sampling_applied'] = 'undersampling'
            
        train_class_distribution_after = dict(sorted(Counter(y_train).items()))
        split_parameter_info['class_distribution_after_sampling'] = train_class_distribution_after
        split_parameter_info['train_samples_after_sampling'] = len(X_train)
        print(f"    - Training data class distribution after sampling: {train_class_distribution_after}")
    else:
        print("    - No sampling applied")
        split_parameter_info['sampling_applied'] = 'None'
        split_parameter_info['train_samples_after_sampling'] = len(X_train)

    # ----------------------------------------------------
    # Step 6: Combine and return the final DataFrames
    # ----------------------------------------------------
    train_data = pd.concat([X_train, y_train], axis=1)
    test_data = pd.concat([X_test, y_test], axis=1)
    
    split_parameter_info['final_train_feature_count'] = train_data.shape[1] - 1
    
    return train_data, test_data, split_parameter_info

In [6]:
# Defining function related to feature selection

def xicor(
    x: npt.ArrayLike,
    y: npt.ArrayLike,
    ties: Union[bool, Literal["auto"]] = "auto",
) -> Tuple[float, float]:
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    n = len(y)

    if len(x) != n:
        raise IndexError(f"x, y length mismatch: {len(x)}, {len(y)}")

    if ties == "auto":
        ties = len(np.unique(y)) < n
    elif not isinstance(ties, bool):
        raise ValueError(
            f'expected ties either "auto" or boolean, '
            f"got {ties} ({type(ties)}) instead"
        )

    y = y[np.argsort(x)]
    r = stats.rankdata(y, method="ordinal")
    nominator = np.sum(np.abs(np.diff(r)))

    if ties:
        l = stats.rankdata(y, method="max")
        denominator = 2 * np.sum(l * (n - l))
        nominator *= n
    else:
        denominator = np.power(n, 2) - 1
        nominator *= 3

    statistic = 1 - nominator / denominator  # upper bound is (n - 2) / (n + 1)
    p_value = stats.norm.sf(statistic, scale=2 / 5 / np.sqrt(n))

    return statistic, p_value
# --- Step-by-step modularization function for variable filtering w. correlation ---
def filter_by_variance(X: pd.DataFrame, var_threshold: float) -> Tuple[pd.DataFrame, Dict]:
    start_time = dt.datetime.now()
    
    X = rescale_df(X)
    
    initial_cols = list(X.columns)
    
    # Calculate the variance for all features.
    features_values_checked = X.var().to_dict()
    
    vt = VarianceThreshold(threshold=var_threshold)
    X_filtered = vt.fit_transform(X)
    vt_mask = vt.get_support()
    vt_cols = X.columns[vt_mask]

    # Store whether each feature was dropped.
    features_dropped_yn = {col: not vt_mask[i] for i, col in enumerate(initial_cols)}
    
    end_time = dt.datetime.now()
    duration = end_time - start_time
    duration_str = str(duration).split('.')[0]
    
    stats = {
        'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'threshold_value': var_threshold,
        'original_count': len(initial_cols),
        'remaining_count': len(vt_cols),
        'features_dropped_yn': features_dropped_yn,
        'features_values_checked': features_values_checked
    }
    print(f"    - Number of features remaining after variance filtering: {stats['remaining_count']}")
    return pd.DataFrame(X_filtered, columns=vt_cols, index=X.index), stats
def filter_by_target_linear_correlation(X: pd.DataFrame, y: pd.Series, threshold: float) -> Tuple[pd.DataFrame, Dict]:
    start_time = dt.datetime.now()
    
    initial_cols = list(X.columns)
    
    # Calculate the correlation between all features and the target.
    correlations = X.corrwith(y).abs()
    features_values_checked = correlations.to_dict()
    
    low_corr_features = correlations[correlations < threshold].index.tolist()
    X_filtered = X.drop(columns=low_corr_features)

    # Store whether each feature was dropped.
    features_dropped_yn = {col: col in low_corr_features for col in initial_cols}
    
    end_time = dt.datetime.now()
    duration = end_time - start_time
    duration_str = str(duration).split('.')[0]
    
    stats = {
        'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'threshold_value': threshold,
        'original_count': X.shape[1],
        'remaining_count': X_filtered.shape[1],
        'features_dropped_yn': features_dropped_yn,
        'features_values_checked': features_values_checked
    }
    print(f"    - Number of features remaining after target linear correlation filtering: {stats['remaining_count']}")
    return X_filtered, stats
def filter_by_target_xicor_correlation(X: pd.DataFrame, y: pd.Series, threshold: float) -> Tuple[pd.DataFrame, Dict]:
    start_time = dt.datetime.now()
    
    # y (타겟)을 수치형 또는 0/1로 변환
    if y.dtype == 'bool':
        y_processed = y.astype(int)
    elif pd.api.types.is_numeric_dtype(y):
        y_processed = y
    else:
        # y가 수치형 또는 불리언이 아닌 경우, 처리가 불가능하므로 오류 반환
        raise TypeError("Target 'y' must be a numeric or boolean type.")

    # X 데이터프레임의 복사본을 만들어 불리언 컬럼을 0/1로 변환
    X_processed = X.copy()
    for col in X_processed.select_dtypes(include='bool').columns:
        X_processed[col] = X_processed[col].astype(int)
    
    to_drop = []
    features_dropped_yn = {}
    features_values_checked = {}
    initial_cols = list(X.columns)
    
    for col in initial_cols:
        # 컬럼이 수치형 또는 불리언인지 확인
        if pd.api.types.is_numeric_dtype(X[col]) or pd.api.types.is_bool_dtype(X[col]):
            try:
                # xicor는 변환된 데이터(X_processed, y_processed)를 사용
                xi_corr_val, p_value = xicor(X_processed[col].values, y_processed.values)
                is_dropped = abs(xi_corr_val) <= threshold
                features_dropped_yn[col] = str(is_dropped)
                features_values_checked[col] = xi_corr_val
                if is_dropped:
                    to_drop.append(col)
            except Exception as e:
                # xicor 계산 오류 시 드롭하지 않음
                print(f"Warning: Failed to calculate xicor for feature '{col}'. Error: {e}")
                features_dropped_yn[col] = 'False'
                features_values_checked[col] = None
        else:
            # 수치형/불리언이 아닌 컬럼은 드롭하지 않고, 정보도 저장하지 않음
            features_dropped_yn[col] = 'False'
    
    X_filtered = X.drop(columns=to_drop)
    
    end_time = dt.datetime.now()
    duration = end_time - start_time
    duration_str = str(duration).split('.')[0]
    
    stats = {
        'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'threshold_value': threshold,
        'original_count': X.shape[1],
        'remaining_count': X_filtered.shape[1],
        'features_dropped_yn': features_dropped_yn,
        'features_values_checked': features_values_checked
    }
    
    print(f"    - 타겟 Xi Cor 필터링 후 남은 피처 수: {stats['remaining_count']}")
    return X_filtered, stats
def filter_by_feature_linear_correlation(X: pd.DataFrame, threshold: float) -> Tuple[pd.DataFrame, Dict]:
    start_time = dt.datetime.now()
    
    features_dropped_yn = {col: False for col in X.columns}
    features_values_checked = {}
    initial_cols = list(X.columns)
    
    to_drop = []
    
    if len(initial_cols) > 1:
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
        
        # Store correlation values for all column pairs
        for i in range(len(upper.columns)):
            for j in range(i + 1, len(upper.columns)):
                col1 = upper.columns[i]
                col2 = upper.columns[j]
                
                # Get the correlation value for 'col1' and 'col2'.
                correlation_value = upper.loc[col1, col2]
                
                # Keep consistency by using a sorted tuple as the key.
                pair = tuple(sorted((col1, col2)))
                features_values_checked[str(pair)] = correlation_value
                
                # Add to the 'to_drop' list if it exceeds the threshold.
                if correlation_value > threshold:
                    if col2 not in to_drop:
                        to_drop.append(col2)
        
        # Update features_dropped_yn based on the 'to_drop' list.
        for col in to_drop:
            features_dropped_yn[col] = True

        X_filtered = X.drop(columns=to_drop)
    else:
        X_filtered = X

    end_time = dt.datetime.now()
    duration = end_time - start_time
    duration_str = str(duration).split('.')[0]
    
    stats = {
        'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'threshold_value': threshold,
        'original_count': X.shape[1],
        'remaining_count': X_filtered.shape[1],
        'features_dropped_yn': features_dropped_yn,
        'features_values_checked': features_values_checked
    }
    
    print(f"    - Remaining features after filtering by linear correlation: {stats['remaining_count']}")
    return X_filtered, stats
def filter_by_feature_xicor_correlation(X: pd.DataFrame, threshold: float) -> Tuple[pd.DataFrame, Dict]:
    start_time = dt.datetime.now()
    
    to_drop = []
    features_dropped_yn = {col: False for col in X.columns}
    features_values_checked = {}
    initial_cols = list(X.columns)
    
    if len(initial_cols) > 1:
        # First, calculate and store the xi correlation for all column pairs.
        for i in range(len(initial_cols)):
            for j in range(i + 1, len(initial_cols)):
                col1 = initial_cols[i]
                col2 = initial_cols[j]
                
                # Calculate the non-linear correlation value using the 'xicor' function.
                # This value is always stored regardless of the threshold.
                xi_corr_val = xicor(X[col1].values, X[col2].values)
                pair_key = str(tuple(sorted((col1, col2))))
                features_values_checked[pair_key] = xi_corr_val
        
        # Now, determine which columns to drop based on the stored values.
        # This loop finds columns with high correlation and adds them to the 'to_drop' list.
        # The logic previously skipped has been removed, and all columns are checked again.
        for i in range(len(initial_cols)):
            for j in range(i + 1, len(initial_cols)):
                col1 = initial_cols[i]
                col2 = initial_cols[j]
                
                # Do not further check features that are already scheduled to be dropped.
                if col1 in to_drop or col2 in to_drop:
                    continue
                
                pair_key = str(tuple(sorted((col1, col2))))
                xi_corr_val = features_values_checked[pair_key] # Use the already calculated value
                
                if xi_corr_val > threshold:
                    # For simplicity, this implementation drops col2.
                    to_drop.append(col2)
                    features_dropped_yn[col2] = True
        
        X_filtered = X.drop(columns=to_drop, axis=1)
    else:
        X_filtered = X
        
    end_time = dt.datetime.now()
    duration = end_time - start_time
    duration_str = str(duration).split('.')[0]
    
    stats = {
        'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'threshold_value': threshold,
        'original_count': X.shape[1],
        'remaining_count': X_filtered.shape[1],
        'features_dropped_yn': features_dropped_yn,
        'features_values_checked': features_values_checked
    }
    
    print(f"    - Remaining features after filtering by non-linear correlation: {stats['remaining_count']}")
    return X_filtered, stats
# --- Change integrated filtering workflow function name ---
def feature_filter(X: pd.DataFrame, y: pd.Series, params: Dict) -> Tuple[pd.DataFrame, list, Dict]:
    X_filtered = X.copy()
    filter_stats = {}
    print("\n--- Starting Feature Filtering ---")

    # 1. Variance Filtering
    if params.get('apply_variance_filter', True):
        X_filtered, stats = filter_by_variance(X_filtered, params['var_threshold'])
        filter_stats['variance'] = stats
    # 2. Linear Correlation Filtering with Target
    if params.get('apply_target_linear_corr_filter', True) and X_filtered.shape[1] > 0:
        X_filtered, stats = filter_by_target_linear_correlation(X_filtered, y, params['target_linear_corr_threshold'])
        filter_stats['target_linear_correlation'] = stats
    # 3. Non-linear Correlation (Xi Cor) Filtering with Target
    if params.get('apply_target_xicor_filter', True) and X_filtered.shape[1] > 0:
        X_filtered, stats = filter_by_target_xicor_correlation(X_filtered, y, params['target_xicor_threshold'])
        filter_stats['target_xicor_correlation'] = stats
    # 4. Linear Correlation Filtering between Features
    if params.get('apply_feature_linear_corr_filter', True) and X_filtered.shape[1] > 1:
        X_filtered, stats = filter_by_feature_linear_correlation(X_filtered, params['feature_linear_corr_threshold'])
        filter_stats['feature_linear_correlation'] = stats
    # 5. Non-linear Correlation (Xi Cor) Filtering between Features
    if params.get('apply_feature_xicor_filter', True) and X_filtered.shape[1] > 1:
        X_filtered, stats = filter_by_feature_xicor_correlation(X_filtered, params['feature_xicor_threshold'])
        filter_stats['feature_xicor_correlation'] = stats
            
    final_cols = list(X_filtered.columns)
    return X_filtered, final_cols, filter_stats
def _get_estimator(estimator_params: Dict) -> BaseEstimator:
    """
    Creates a Scikit-learn estimator object based on the given parameter dictionary.
    """
    estimator_name = estimator_params.get("name")
    params = estimator_params.get("params", {})
    
    if estimator_name == "LogisticRegression":
        return LogisticRegression(random_state=42, n_jobs=-1, **params)
    elif estimator_name == "RandomForestClassifier":
        return RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    elif estimator_name == "LGBMClassifier":
        return LGBMClassifier(random_state=42, n_jobs=-1, **params)
    else:
        raise ValueError(f"Unsupported estimator: {estimator_name}")
# --- Add helper function for extracting variable importance ---
def _get_feature_importances(estimator: BaseEstimator, feature_names: List[str]) -> Dict[str, float]:
    """
    Helper function to extract variable importances from an estimator.
    """
    importances = {}
    if hasattr(estimator, 'feature_importances_'):
        importances = {col: imp for col, imp in zip(feature_names, estimator.feature_importances_)}
    elif hasattr(estimator, 'coef_'):
        # For models like LogisticRegression, use the coef_ attribute
        coefs = estimator.coef_[0] if estimator.coef_.ndim > 1 else estimator.coef_
        importances = {col: abs(coef) for col, coef in zip(feature_names, coefs)}
    return importances
def run_model_based_feature_selection(
    X_in: pd.DataFrame, 
    y: pd.Series, 
    selector_name: str, 
    selector_params: Dict
) -> Tuple[List[str], Dict]:
    """
    **A general function to selectively run RFE, SFS, or SelectFromModel.**
    
    Args:
        X (pd.DataFrame): Feature data
        y (pd.Series): Target data
        selector_name (str): Name of the variable selector to use ('RFE', 'SFS', 'SFM')
        selector_params (Dict): Dictionary of parameters to pass to the variable selector

    Returns:
        Tuple[List[str], Dict]: A list of selected features and a dictionary of statistics
    """
    start_time = dt.datetime.now()
    
    X = X_in.copy()
    
    initial_features = list(X.columns)
    
    estimator_params = selector_params.get("estimator")
    estimator = _get_estimator(estimator_params)
    
    selected_features = []
    dropped_features = []
    stats = {}
    
    if selector_name == 'RFE':
        n_features_to_select = selector_params.get('n_features_to_select')
        step = selector_params.get('step', 1)
        
        selector = RFE(estimator=estimator, n_features_to_select=n_features_to_select, step=step)
        selector.fit(X, y)
        selected_mask = selector.get_support()
        selected_features = list(X.columns[selected_mask])
        
        # RFE provides a ranking. Add the ranking to stats.
        ranking = {col: rank for col, rank in zip(initial_features, selector.ranking_)}
        stats = {'method': 'RFE', 'n_features_to_select': n_features_to_select, 'step': step, 'ranking': ranking}

        # Extract importances of selected features and add to stats.
        if ranking and any(rank == 1 for rank in ranking.values()):
            selected_estimator = _get_estimator(estimator_params)
            selected_estimator.fit(X[selected_features], y)
            importances = _get_feature_importances(selected_estimator, selected_features)
            stats['importances'] = importances
            
    elif selector_name == 'SFM':
        threshold = selector_params.get('threshold', 'median')
        
        # First, fit the model and then pass it to SelectFromModel.
        estimator.fit(X, y)
        selector = SelectFromModel(estimator, prefit=True, threshold=threshold)
        selected_mask = selector.get_support()
        selected_features = list(X.columns[selected_mask])

        # Extract variable importances from the model.
        importances = _get_feature_importances(estimator, initial_features)
        stats = {'method': 'SFM', 'threshold': threshold, 'importances': importances}

    elif selector_name == 'SFS':
        n_features_to_select = selector_params.get('n_features_to_select')
        direction = selector_params.get('direction', 'forward')
        
        selector = SFS(estimator=estimator, n_features_to_select=n_features_to_select, direction=direction, cv=5)
        
        selector.fit(X, y)
        selected_features = list(X.columns[selector.get_support()])
        
        # SFS does not directly provide importances, so re-fit the model with selected variables to extract them.
        selected_estimator = _get_estimator(estimator_params)
        selected_estimator.fit(X[selected_features], y)
        importances = _get_feature_importances(selected_estimator, selected_features)

        stats = {
            'method': 'SFS', 
            'n_features_to_select': n_features_to_select, 
            'direction': direction,
            'importances': importances
        }
        
    else:
        raise ValueError(f"Unsupported selector name: {selector_name}. Choose from 'RFE', 'SFS', 'SFM'.")

    dropped_features = [col for col in initial_features if col not in selected_features]

    end_time = dt.datetime.now()
    duration_str = str(end_time - start_time).split('.')[0]

    stats.update({
        'start_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'duration': duration_str,
        'original_count': len(initial_features),
        'remaining_count': len(selected_features),
        'selected_features': selected_features,
        'dropped_features': dropped_features,
    })
    
    print(f"--- {selector_name} Selector completed ---")
    print(f"Number of remaining features: {stats['remaining_count']}")
    
    return selected_features, stats
# --- Feature Selection Function ---
def select_feature(train_data: pd.DataFrame, feature_selector_params: Dict) -> Dict:
    """
    Performs the feature selection workflow and returns the results in a dictionary.
    """
    X_train = train_data.iloc[:, :-1]
    y_train = train_data.iloc[:, -1]
    feature_selection_info={}
    feature_selection_info = feature_selector_params.copy()
    initial_feature_count = X_train.shape[1]
    
    selector_name = feature_selector_params.get("feature_selector_name")
    final_features = list(X_train.columns)
    stats = {}
    
    print(f"\n--- Feature Selector: {selector_name} ---")

    try:
        if selector_name == 'FeatureFilter':
            filter_params = feature_selector_params['filter_methods']
            _, final_features, stats = feature_filter(X=X_train, y=y_train, params=filter_params)
        
        elif selector_name in ['RFE', 'SFM', 'SFS']:
            # selector_params = feature_selector_params.get(f"{selector_name.lower()}_params")
            selector_params = feature_selector_params.get("params")
            if not selector_params:
                raise KeyError(f"Missing '{selector_name.lower()}_params' in configuration.")
            
            final_features, stats = run_model_based_feature_selection(
                X_train,
                y_train,
                selector_name=selector_name,
                selector_params=selector_params
            )
        else:
            print("\n--- Feature Selector: No-op ---")
            stats = {'method': 'No-op', 'original_count': initial_feature_count, 'remaining_count': initial_feature_count, 'selected_features': final_features, 'dropped_features': []}
    
    except Exception as e:
        print(f"An error occurred during feature selection: {e}")
        # In case of an error, return original features
        stats = {'method': 'Error', 'original_count': initial_feature_count, 'remaining_count': initial_feature_count, 'selected_features': final_features, 'dropped_features': [], 'error_message': str(e)}

    final_feature_count = len(final_features)
    feature_selection_info['initial_feature_count'] = initial_feature_count
    feature_selection_info['final_feature_count'] = final_feature_count
    feature_selection_info['final_features'] = final_features
    feature_selection_info['selection_details'] = stats

    # (Modify save logic)
    destination_dir = 'result/feature_selection/jsons'
    os.makedirs(destination_dir, exist_ok=True)
    current_time = dt.datetime.now()
    file_id = uuid.uuid4().hex[:8]
    filename = "feature_selection_info_" + current_time.strftime('%y%m%d_%H%M%S') + f'_{file_id}.json'
    file_path = os.path.join(destination_dir, filename)

    with open(file_path, 'w', encoding='utf-8') as f:
        # Use a class encoder to convert numpy int64 to Python int
        json.dump(feature_selection_info, f, indent=4, ensure_ascii=False, cls=NpEncoder)

    print(f"\nFeature selection results saved to '{file_path}' file.")
    print(f"\n- Final feature count: {final_feature_count}")
    
    feature_selection_info['feature_selection_info_json_path'] = file_path
    
    return feature_selection_info

In [7]:
# Functions to train train_model

# Function to train a Random Forest model with cross-validation and hyperparameter tuning
def train_model_rf_cv(train_dataset: pd.DataFrame, feature_selection_info: dict, train_parameters: dict = None):
    print("      Training the Random Forest model with cross-validation & hyperparameter tuning...\n")
    
    model_parameter_info = {}
    
    X, y = train_dataset.iloc[:, :-1], train_dataset.iloc[:, -1]
    final_features = feature_selection_info['final_features']
    X = X[final_features]

    # Set parameter grid and GridSearchCV parameters based on train_parameters
    if train_parameters and train_parameters.get('function_name') == 'train_model_rf_cv':
        param_grid = train_parameters.get('param_grid', {})
        cv = train_parameters.get('cv', 3)
        verbose = train_parameters.get('verbose', 1)
        
        # Apply f2_rare_scorer setting logic
        scoring_params = train_parameters.get('f2_rare_scorer', {})
        if scoring_params.get('name') == 'fbeta_score':
            beta = scoring_params.get('beta', 2)
            pos_label = scoring_params.get('pos_label', 1)
            scorer = make_scorer(fbeta_score, beta=beta, pos_label=pos_label)
        else:
            scorer = make_scorer(f2_rare_scorer, greater_is_better=True)
            
    else:
        param_grid = {
            "n_estimators": [20, 50, 100],
            "max_depth": [2, 5, None],
            "min_samples_split": [2, 5],
        }
        cv = 3
        verbose = 1
        scorer = make_scorer(f2_rare_scorer, greater_is_better=True)
    
    # Store configured parameter information
    model_parameter_info['param_grid'] = param_grid
    model_parameter_info['cv'] = cv
    model_parameter_info['verbose'] = verbose
    if 'f2_rare_scorer' in locals():
        model_parameter_info['f2_rare_scorer'] = {
            'name': 'fbeta_score',
            'beta': scorer._kwargs.get('beta'),
            'pos_label': scorer._kwargs.get('pos_label')
        }

    # Assign class weights to solve the class imbalance problem
    # n_neg = len(y) - sum(y)
    n_neg = len(y) - sum(int(i) for i in y)

    # If the value is None or the key is not in the dictionary, set to n_neg
    if train_parameters.get('class_weight_multiplier') == '':
        class_weight_multiplier = n_neg
    else:
        class_weight_multiplier = eval(train_parameters.get('class_weight_multiplier'))
        
    class_weight = {0: 1, 1: class_weight_multiplier}

    model_parameter_info['class_weight_multiplier'] = train_parameters.get('class_weight_multiplier')    
    
    rf_model = RandomForestClassifier(class_weight=class_weight, random_state=42)

    grid_search = GridSearchCV(
        estimator=rf_model,
        param_grid=param_grid,
        scoring=scorer,
        cv=cv,
        verbose=verbose,
        n_jobs=-1,
    )

    grid_search.fit(X, y)
    best_model = grid_search.best_estimator_

    print(f"\n    Best parameters found: {grid_search.best_params_}")
    print(f"    Best F2 (class=1) score (CV): {grid_search.best_score_:.4f}\n")
    
    model_parameter_info['best_params'] = grid_search.best_params_

    importance_dict = {
        "Features": X.columns,
        "Importance": best_model.feature_importances_,
        "Importance_abs": np.abs(best_model.feature_importances_),
    }
    importance = pd.DataFrame(importance_dict).sort_values(
        by="Importance", ascending=True
    )
    
    return best_model, importance, model_parameter_info
# Function to train an XGBoost model with cross-validation and hyperparameter tuning
def train_model_xgboost_cv(train_dataset: pd.DataFrame, feature_selection_info: dict, train_parameters: dict = None):
    print(
        "     Training the XGBoost model with cross-validation & hyperparameter tuning...\n"
    )

    model_parameter_info = {}

    X, y = train_dataset.iloc[:, :-1], train_dataset.iloc[:, -1]

    final_features = feature_selection_info['final_features']
    X = X[final_features]

    # Set parameters based on the algorithm name in train_parameters
    if train_parameters and train_parameters.get('function_name') == 'train_model_xgboost_cv':
        param_grid = train_parameters.get('param_grid', {})
        scale_pos_weight_multiplier = train_parameters.get('scale_pos_weight_multiplier', 2)
        cv = train_parameters.get('cv', 3)
        verbose = train_parameters.get('verbose', 1)
        
        # Apply logic for f2_rare_scorer
        scoring_params = train_parameters.get('f2_rare_scorer', {})
        if scoring_params.get('name') == 'fbeta_score':
            beta = scoring_params.get('beta', 2)
            pos_label = scoring_params.get('pos_label', 1)
            f2_rare_scorer = make_scorer(fbeta_score, beta=beta, pos_label=pos_label)
        else:
            # Default F2 score
            f2_rare_scorer = make_scorer(lambda y_true, y_pred: fbeta_score(y_true, y_pred, beta=2, pos_label=1))
            
    else:
        # Default hyperparameter settings
        param_grid = {
            "n_estimators": [30, 50, 100, 200],
            "max_depth": [2, 5],
            "learning_rate": [0.01, 0.1, 0.2],
        }
        scale_pos_weight_multiplier = 2
        cv = 3
        verbose = 1
        f2_rare_scorer = make_scorer(lambda y_true, y_pred: fbeta_score(y_true, y_pred, beta=2, pos_label=1))

    # Save the configured parameter information
    model_parameter_info['param_grid'] = param_grid
    model_parameter_info['scale_pos_weight_multiplier'] = scale_pos_weight_multiplier
    model_parameter_info['cv'] = cv
    model_parameter_info['verbose'] = verbose
    model_parameter_info['f2_rare_scorer'] = {
        'name': 'fbeta_score',
        'beta': f2_rare_scorer._kwargs.get('beta'),
        'pos_label': f2_rare_scorer._kwargs.get('pos_label')
    }

    # Calculate 'scale_pos_weight' for class imbalance
    n_pos = sum(y)
    n_neg = len(y) - n_pos
    scale_pos_weight = n_neg / n_pos * scale_pos_weight_multiplier if n_pos > 0 else 1

    # Create the XGBoost model object
    xgb_model = XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight,
    )

    # Configure GridSearchCV
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        scoring=f2_rare_scorer,
        cv=cv,
        verbose=verbose,
        n_jobs=-1,
    )

    grid_search.fit(X, y)
    best_model = grid_search.best_estimator_

    print(f"\n     Best parameters found: {grid_search.best_params_}")
    print(f"     Best F2 (class=1) score: {grid_search.best_score_:.4f}\n")
    
    model_parameter_info['best_params'] = grid_search.best_params_

    # Calculate feature importances for the best model
    importance_dict = {
        "Features": X.columns,
        "Importance": best_model.feature_importances_,
        "Importance_abs": np.abs(best_model.feature_importances_),
    }
    importance = pd.DataFrame(importance_dict).sort_values(
        by="Importance", ascending=True
    )

    return best_model, importance, model_parameter_info

In [8]:
# Function to perform predictions on the test dataset
def forecast(test_dataset: pd.DataFrame, trained_model, feature_selection_info: dict):
    print("     Forecasting the test dataset...")
    X = test_dataset.iloc[:, :-1]

    # If not a baseline model, reconstruct X by loading features from 'final_features.json'.
    if not isinstance(trained_model, BaselineModel):
        # with open("final_features.json", "r") as f:
        #     final_features = json.load(f)
        final_features = feature_selection_info['final_features']
        X = X[final_features]


    # Get the prediction probabilities for class 1.
    predictions = trained_model.predict_proba(X)[:, 1]
    print("     Forecasting done!")

    # Use SHAP to analyze the explainability of model predictions.
    # Use TreeExplainer for tree-based models, otherwise use KernelExplainer.
    if hasattr(trained_model, "feature_importances_"):
        explainer = shap.TreeExplainer(trained_model)
    elif not isinstance(trained_model, BaselineModel):
        explainer = shap.Explainer(trained_model, X)
    
    # Calculate SHAP values only if it's not a baseline model.
    if not isinstance(trained_model, BaselineModel):
        shap_values = explainer(X)
        # (Commented-out code) Plot SHAP summary plot.
        # plt.figure(figsize=(10, 5))
        # shap.summary_plot(shap_values, X, max_display=10, show=False)
        # plt.show()
    else:
        shap_values = None

    return predictions, [shap_values, X]


In [9]:
# Function to find the optimal threshold that maximizes the F2 score
def find_best_threshold(best_model, train_dataset, feature_selection_info: dict):
    """
    Finds the optimal threshold for classification that maximizes the F2 score.

    Parameters:
    - best_model: A trained classifier model with a `predict_proba` method.
    - train_dataset: DataFrame containing features and the target.

    Returns:
    - best_threshold: The optimal threshold that maximizes the F2 score.
    """
    X, y = train_dataset.iloc[:, :-1], train_dataset.iloc[:, -1]

    # If not a baseline model, reconstruct X by loading features from 'final_features.json'
    if not isinstance(best_model, BaselineModel):
        # with open("final_features.json", "r") as f:
        #     final_features = json.load(f)
        final_features = feature_selection_info['final_features']
        X = X[final_features]

    # Get the probability of class 1 (fail) predicted by the model.
    prob_class1 = best_model.predict_proba(X)[:, 1]

    # Try 100 threshold candidates from 0 to 1.
    thresholds = np.linspace(0, 1, 100)
    f2_scores = []

    for threshold in thresholds:
        y_pred = (prob_class1 >= threshold).astype(int) # Generate predicted labels based on the threshold
        # score = fbeta_score(y, y_pred, beta=2, pos_label=1) # Calculate F2 score
        score = fbeta_score(y, y_pred, beta=4, pos_label=1) # Calculate F2 score
        f2_scores.append(score)

    # Find the threshold that recorded the highest F2 score.
    best_idx = np.argmax(f2_scores)
    best_threshold = thresholds[best_idx]
    best_f2_score = f2_scores[best_idx]

    print(
        f"Best threshold for F2 score: {best_threshold:.4f} with F2 score: {best_f2_score:.4f}"
    )

    # ------------------------------------------------------------------
    # Apply the user-selected threshold.
    # ------------------------------------------------------------------

    # Add 'Probability' and 'Historical' columns to the training dataset.
    train_dataset["Probability"] = prob_class1
    train_dataset["Historical"] = y
    return train_dataset, best_threshold


In [10]:
# Function to calculate the ROC curve from scratch
def roc_from_scratch(probabilities, test_dataset, partitions=100):
    print("     Calculation of the ROC curve...")
    y_test = test_dataset.iloc[:, -1] # Actual labels of the test data

    roc = []
    # Iterate through 101 thresholds from 0 to 1.
    for i in range(partitions + 1):
        thr = i / partitions
        threshold_vector = (probabilities >= thr).astype(int) # Predict based on the threshold
        tpr, fpr = true_false_positive(threshold_vector, y_test) # Calculate TPR and FPR
        roc.append([fpr, tpr])

    # Create a DataFrame with the calculated TPR and FPR.
    roc_data = pd.DataFrame(roc, columns=["False positive rate", "True positive rate"])
    print("     Calculation done")
    print("     Scoring...")

    # Calculate the AUC score using scikit-learn's 'roc_auc_score'.
    auc_score = roc_auc_score(y_test, probabilities)
    print("     Scoring done\n")
    return roc_data, auc_score


In [11]:
# Function to create confusion matrix metrics on the training dataset
def create_metrics_on_train(train_dataset, threshold):
    """
    After training, predicts on the training dataset with a given threshold (for class 1, fail).
    """
    # ------------------------------------------------------------------
    # Apply the user-selected threshold.
    # ------------------------------------------------------------------
    # Predict 1 if 'Probability' is greater than or equal to the threshold, otherwise 0.
    forecast = (train_dataset["Probability"] >= threshold).astype(int)

    train_dataset["Forecast"] = forecast
    # Apply confusion matrix labels.
    train_dataset["True/False/Positive/Negative"] = train_dataset.apply(
        _confusion_label, axis=1
    )
    return train_dataset


In [12]:
# Function to generate various performance metrics based on prediction results
def create_metrics(
    predictions: np.array, test_dataset: pd.DataFrame, auc_score, threshold
):
    print("     Creating the metrics...")
    # Generate final predicted labels based on the threshold.
    threshold_vector = (predictions >= threshold).astype(int)

    y_test = test_dataset.iloc[:, -1]

    # Calculate TP, TN, FP, FN values.
    tp = ((threshold_vector == 1) & (y_test == 1)).sum()
    tn = ((threshold_vector == 0) & (y_test == 0)).sum()
    fp = ((threshold_vector == 1) & (y_test == 0)).sum()
    fn = ((threshold_vector == 0) & (y_test == 1)).sum()

    # Calculate F1 score (for class 1)
    denom = 2 * tp + fp + fn
    if denom == 0:
        f1_score = 0.0
    else:
        f1_score = 2 * tp / denom
    f1_score = np.around(f1_score, 2) # Round to two decimal places

    # Calculate Accuracy
    accuracy = np.around((tp + tn) / (tp + tn + fp + fn + 1e-9), 2)
    # Round AUC score
    auc_score = np.around(auc_score, 2)

    # Store TP, TN, FP, FN values in a dictionary.
    dict_ftpn = {"tp": tp, "tn": tn, "fp": fp, "fn": fn}
    number_of_good_predictions = tp + tn
    number_of_false_predictions = fp + fn

    # Calculate Precision and Recall
    if (tp + fp) == 0:
        precision = 0.0
    else:
        precision = tp / (tp + fp)
    precision = np.around(precision, 2)

    if (tp + fn) == 0:
        recall = 0.0
    else:
        recall = tp / (tp + fn)
    recall = np.around(recall, 2)

    # Return all metrics in a dictionary.
    metrics = {
        "f1_score": f1_score,
        "recall": recall,
        "precision": precision,
        "accuracy": accuracy,
        "auc_score": auc_score,
        "dict_ftpn": dict_ftpn,
        "number_of_predictions": len(predictions),
        "number_of_good_predictions": number_of_good_predictions,
        "number_of_false_predictions": number_of_false_predictions,
    }

    return metrics


In [13]:
# Function to generate various performance metrics based on prediction results
def create_metrics(
    predictions: np.array, test_dataset: pd.DataFrame, auc_score, threshold
):
    print("     Creating the metrics...")
    # Generate final predicted labels based on the threshold.
    threshold_vector = (predictions >= threshold).astype(int)

    y_test = test_dataset.iloc[:, -1]

    # Calculate TP, TN, FP, FN values.
    tp = ((threshold_vector == 1) & (y_test == 1)).sum()
    tn = ((threshold_vector == 0) & (y_test == 0)).sum()
    fp = ((threshold_vector == 1) & (y_test == 0)).sum()
    fn = ((threshold_vector == 0) & (y_test == 1)).sum()

    # Calculate F1 score (for class 1)
    denom = 2 * tp + fp + fn
    if denom == 0:
        f1_score = 0.0
    else:
        f1_score = 2 * tp / denom
    f1_score = np.around(f1_score, 2) # Round to two decimal places

    # Calculate Accuracy
    accuracy = np.around((tp + tn) / (tp + tn + fp + fn + 1e-9), 2)
    # Round AUC score
    auc_score = np.around(auc_score, 2)

    # Store TP, TN, FP, FN values in a dictionary.
    dict_ftpn = {"tp": tp, "tn": tn, "fp": fp, "fn": fn}
    number_of_good_predictions = tp + tn
    number_of_false_predictions = fp + fn

    # Calculate Precision and Recall
    if (tp + fp) == 0:
        precision = 0.0
    else:
        precision = tp / (tp + fp)
    precision = np.around(precision, 2)

    if (tp + fn) == 0:
        recall = 0.0
    else:
        recall = tp / (tp + fn)
    recall = np.around(recall, 2)

    # Return all metrics in a dictionary.
    metrics = {
        "f1_score": f1_score,
        "recall": recall,
        "precision": precision,
        "accuracy": accuracy,
        "auc_score": auc_score,
        "dict_ftpn": dict_ftpn,
        "number_of_predictions": len(predictions),
        "number_of_good_predictions": number_of_good_predictions,
        "number_of_false_predictions": number_of_false_predictions,
    }

    return metrics


In [14]:
# Function to organize prediction results into a DataFrame
def create_results(forecast_values, test_dataset, threshold):
    # Create a series of predicted probabilities, rounded to two decimal places.
    forecast_series_proba = pd.Series(
        np.around(forecast_values, decimals=2),
        index=test_dataset.index,
        name="Probability",
    )
    # Create a series of predicted labels (0 or 1) based on the threshold.
    forecast_series = pd.Series(
        (forecast_values > threshold).astype(int),
        index=test_dataset.index,
        name="Forecast",
    )
    # Create a series of actual labels.
    true_series = pd.Series(
        test_dataset.iloc[:, -1], name="Historical", index=test_dataset.index
    )
    # Create a series containing the index numbers.
    index_series = pd.Series(
        range(len(true_series)), index=test_dataset.index, name="Id"
    )

    # Combine all series into a single DataFrame.
    results = pd.concat(
        [index_series, forecast_series_proba, forecast_series, true_series], axis=1
    )
    # Add confusion matrix labels.
    results["True/False/Positive/Negative"] = results.apply(_confusion_label, axis=1)
    return results


In [15]:
# Define the required pipeline functions

# feature select test : pipeline
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_test_rf(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


In [16]:
# Define default parameters

# Train_test_data_separation default parameters
split_parameter_default = \
    {
        'test_size': 0.2,
        'random_state': 42,
        # Filters applied directly in create_train_test_data
        'apply_filter_split': False,
        'var_threshold_split': 0.0,
        'corr_threshold_split': 0.98,
        
        'sampling_ratio': None,
        
        'apply_feature_generation': False,
        'sum_features': False,  # Whether to create pairwise sum features
        'diff_features': False,  # Whether to create pairwise difference features
        'poly_features': False,  # Whether to create polynomial features
        'poly_degree': 2,

        # Filters applied within the feature_generator
        'apply_filter_gen': False,
        'var_threshold_gen': 0.0,
        'corr_threshold_gen': 0.1,
    }



# Default parameters related to feature selection
# --- Applying a Filter Method (FeatureFilter)
feature_selector_params_FeatureFilter_default = {
    "feature_selector_name": "FeatureFilter",
    "filter_methods": {
        "apply_variance_filter": False,
        "var_threshold": 0.01,
        "apply_target_linear_corr_filter": False,
        "target_linear_corr_threshold": 0.05,
        "apply_target_xicor_filter": False,
        "target_xicor_threshold": 0.1,
        "apply_feature_linear_corr_filter": False,
        "feature_linear_corr_threshold": 0.95,
        "apply_feature_xicor_filter": False,
        "feature_xicor_threshold": 0.9
    }
}
feature_selector_params_estimator_LogisticRegression_default = \
{
    "name": "LogisticRegression"
}
feature_selector_params_estimator_RandomForestClassifier_default = \
{
    "name": "RandomForestClassifier",
    "params": {
        # "n_estimators": 150, # Number of trees
        "n_estimators": 250, # Number of trees
        # "max_depth": 10 # Maximum depth of trees
        "max_depth": 12 # Maximum depth of trees
    }
}
feature_selector_params_estimator_LGBMClassifier_default = \
{
    "name": "LGBMClassifier",
    "params": {
        "n_estimators": 200, # Number of boosting stages
        "learning_rate": 0.05 # Learning rate
    }
}
# --- SFM (SelectFromModel) - Model-based feature selection
feature_selector_params_sfm_default = \
{
    "feature_selector_name": "SFM",
    "params": {
        "threshold": "median", # Feature importance threshold
        # "estimator": feature_selector_params_estimator_LGBMClassifier_default
        "estimator": feature_selector_params_estimator_RandomForestClassifier_default
    },
    "filter_methods": "apply_SelectFromModel_filter"
}
# --- SFS (Sequential Feature Selector)
feature_selector_params_sfs_default = \
{
    "feature_selector_name": "SFS",
    "params": {
        "n_features_to_select": "auto", # Number of features to select (auto)
        "direction": "forward", # Forward selection ('forward') or backward elimination ('backward')
        "scoring": "accuracy", # Model performance metric
        "estimator": feature_selector_params_estimator_RandomForestClassifier_default
    },
    "filter_methods": "apply_SequentialFeatureSelector_filter"
}
# --- In case feature selection is not performed
feature_selector_params_no_op = \
{
    "feature_selector_name": "None"
}


# Training-related default parameters
train_parameters_list_default = \
{'baseline': {'function_name': 'train_model_baseline',
              'importance_data': {'Features': ['SensorOffsetHot-Cold',
                                               'band gap dpat_ok for band gap',
                                               'Radius'],
                                  'Importance': [56.6, 4.65, 96.9]}},
 'logistic_regression_cv': {'cv': 3,
                            'f2_rare_scorer': {'beta': 2,
                                               'name': 'fbeta_score',
                                               'pos_label': 1},
                            'function_name': 'train_model_logistic_regression_cv',
                            'param_grid': {'C': [0.0001,
                                                 0.001,
                                                 0.01,
                                                 0.1,
                                                 1,
                                                 10,
                                                 20],
                                           'solver': ['lbfgs', 'liblinear']}},
 'logistic_regression': {'f2_rare_scorer': {'beta': 2,
                                            'name': 'fbeta_score',
                                            'pos_label': 1},
                         'function_name': 'train_model_logistic_regression',
                         'n_trials': 50,
                         'param_ranges': {'C': [0.0001, 20],
                                          'class_weight_multiplier': [1,
                                                                      20],
                                          'max_iter': 10,
                                          'solver': ['liblinear',
                                                     'saga']}},
 'rf_cv': {'class_weight_multiplier': 'len(y) - sum(y)',
           'cv': 3,
           'f2_rare_scorer': {'beta': 2, 'name': 'fbeta_score', 'pos_label': 1},
           'function_name': 'train_model_rf_cv',
           'param_grid': {'max_depth': [2, 5, None],
                          'min_samples_split': [2, 5],
                          'n_estimators': [20, 50, 100]},
           'verbose': 1},
 'random_forest': {'cv': 5,
                   'f2_rare_scorer': {'beta': 2,
                                      'name': 'fbeta_score',
                                      'pos_label': 1},
                   'function_name': 'train_model_rf_optuna',
                   'n_trials': 50,
                   'param_ranges': {'max_depth': {'high': 30, 'low': 10},
                                    'max_features': {'choices': ['sqrt', 0.5, 0.8]},
                                    'min_samples_leaf': {'high': 10, 'low': 1},
                                    'min_samples_split': {'high': 20, 'low': 2},
                                    'n_estimators': {'high': 300, 'low': 100}}},
 'rf_optuna': {'cv': 5,
               'f2_rare_scorer': {'beta': 2,
                                  'name': 'fbeta_score',
                                  'pos_label': 1},
               'function_name': 'train_model_rf_optuna',
               'n_trials': 50,
               'param_ranges': {'max_depth': {'high': 30, 'low': 10},
                                'max_features': {'choices': ['sqrt', 0.5, 0.8]},
                                'min_samples_leaf': {'high': 10, 'low': 1},
                                'min_samples_split': {'high': 20, 'low': 2},
                                'n_estimators': {'high': 300, 'low': 100}}},
 'xgboost_cv': {'cv': 3,
                'f2_rare_scorer': {'beta': 2,
                                   'name': 'fbeta_score',
                                   'pos_label': 1},
                'function_name': 'train_model_xgboost_cv',
                'param_grid': {'learning_rate': [0.01, 0.1, 0.2],
                               'max_depth': [2, 5],
                               'n_estimators': [30, 50, 100]},
                'scale_pos_weight_multiplier': 2,
                'verbose': 1},
 'xgboost': {'cv': 5,
             'f2_rare_scorer': {'beta': 2,
                                'name': 'fbeta_score',
                                'pos_label': 1},
             'function_name': 'train_model_xgboost_optuna',
             'n_trials': 50,
             'param_ranges': {'colsample_bytree': {'high': 1.0,
                                                   'low': 0.7},
                              'gamma': {'high': 0.5, 'low': 0.1},
                              'learning_rate': {'high': 0.2,
                                                'low': 0.01},
                              'max_depth': {'high': 20, 'low': 5},
                              'n_estimators': {'high': 300, 'low': 100},
                              'reg_alpha': {'high': 0.1, 'low': 1e-06},
                              'reg_lambda': {'high': 0.1, 'low': 1e-06},
                              'subsample': {'high': 1.0, 'low': 0.7}},
             'ratio_multiplier_range': {'high': 1.2, 'low': 0.8}},
 'xgboost_optuna': {'cv': 5,
                    'f2_rare_scorer': {'beta': 2,
                                       'name': 'fbeta_score',
                                       'pos_label': 1},
                    'function_name': 'train_model_xgboost_optuna',
                    'n_trials': 50,
                    'param_ranges': {'colsample_bytree': {'high': 1.0,
                                                          'low': 0.7},
                                     'gamma': {'high': 0.5, 'low': 0.1},
                                     'learning_rate': {'high': 0.2,
                                                       'low': 0.01},
                                     'max_depth': {'high': 20, 'low': 5},
                                     'n_estimators': {'high': 300, 'low': 100},
                                     'reg_alpha': {'high': 0.1, 'low': 1e-06},
                                     'reg_lambda': {'high': 0.1, 'low': 1e-06},
                                     'subsample': {'high': 1.0, 'low': 0.7}},
                    'ratio_multiplier_range': {'high': 1.2, 'low': 0.8}}
}

# prepare initial_dataset

In [17]:
# Prepare source data

data_row_1 = pd.read_excel('1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx')
data_row_2 = pd.read_excel('1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx')

cols_to_keep_df = pd.read_csv('cols_to_keep.csv')
cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()
data_row_1 = data_row_1[cols_to_keep]

# data_row <= data_row1 data_row2
# Select only the columns to join from data_row_2
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                   'BG pass/fail']

# Create a subset DataFrame of data_row_2 with the selected columns
data_row_2_subset = data_row_2[columns_to_join]
# Merge the selected columns from data_row_2 into data_row_1 using the 'DevID' join key
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')
data_row.to_pickle("data_row.p")

data_row = pd.read_pickle('data_row.p')

In [18]:
# Preparing initial data

initial_dataset = data_row.copy() # Copy the original dataset

# prepare for target
initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

# prepare for base model
initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# Create a dictionary for column name changes
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}
initial_dataset.rename(columns=new_column_names, inplace=True)

initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

# Calculate the Radius column
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]
initial_dataset.drop(columns=columns_to_drop, inplace=True)

# preprocess_dataset


In [19]:
# Handling Missing Values

# print("\n      Preprocessing dataset...")
processed_dataset = initial_dataset.copy() # Copy the original dataset

# Create and fit a SimpleImputer object.
# We use the 'mean' strategy to replace missing values with the mean.
# Other strategies (median, most_frequent, etc.) can also be used.
# Before applying the Imputer, remove columns where all values are NaN.

processed_dataset.dropna(axis=1, how='all', inplace=True) # Remove columns where all values are NaN.

# 1. Separate numeric and non-numeric (categorical) features.
numeric_cols = processed_dataset.select_dtypes(include=np.number).columns.tolist() # Select numeric data type columns and convert to a list.
categorical_cols = processed_dataset.select_dtypes(exclude=np.number).columns.tolist() # Select non-numeric data type columns and convert to a list.
# For all columns that can be converted to numeric, coerce non-convertible values to NaN.
for col in processed_dataset.columns: # Iterate through all columns of the dataset.
    # Use pd.to_numeric with the errors='coerce' option for forced conversion.
    # This operation is applied to a copy of the original data.
    processed_dataset[col] = pd.to_numeric(processed_dataset[col], errors='coerce').fillna(processed_dataset[col]) # Replace non-numeric values with NaN, and keep original values.

# 2. Create an object for handling missing values (numeric only).
# Now that all numeric columns are clean, the imputer will work correctly.
numeric_imputer = SimpleImputer(missing_values=np.nan, strategy='mean') # Create an Imputer object to fill NaNs with the mean.

# 3. Apply the Imputer only to the numeric group.
if numeric_cols: # If numeric columns exist, execute the following.
    imputed_numeric_data_array = numeric_imputer.fit_transform(processed_dataset[numeric_cols]) # Fill missing values in numeric columns with the mean.
    # Ensure the number of columns matches
    if imputed_numeric_data_array.shape[1] == len(numeric_cols): # Check if the number of columns in the transformed data matches the original number of numeric columns.
        imputed_numeric_data = pd.DataFrame( # Convert the transformed data to a DataFrame.
            imputed_numeric_data_array,
            columns=numeric_cols,
            index=processed_dataset.index
        )
    else: # If the number of columns doesn't match, print an error message.
        print("Error: Number of columns in imputed data does not match numeric columns.")
        imputed_numeric_data = pd.DataFrame(index=processed_dataset.index)
else: # If there are no numeric columns, create an empty DataFrame.
    imputed_numeric_data = pd.DataFrame(index=processed_dataset.index)

# Use the original data for non-numeric features as is
imputed_categorical_data = processed_dataset[categorical_cols].copy() # Copy non-numeric columns to use them.

# 4. Recombine the processed features based on the index.
# Ensure the indices match before joining
imputed_categorical_data.index = imputed_numeric_data.index # Make the indices of the two DataFrames consistent.
processed_dataset = imputed_numeric_data.join(imputed_categorical_data) # Combine the imputed numeric data and the non-numeric data.
# Verify the final result
# print("--- Modified Result ---")

In [20]:
# Target data redefinition: 1 is fail

# NOTE: Originally, "Pass/Fail_pass=1 => pass, Pass/Fail_pass=0 => fail"
# We invert this to make "1 => fail". In other words, "fail = 1 - old_pass_value".
# old_pass_value = processed_dataset["Pass/Fail_pass"] (1 for pass, 0 for fail)
# new fail => 1 - old_pass_value
processed_dataset["Pass/Fail"] = 1 - processed_dataset["Pass/Fail_pass"] # Invert the 'Pass/Fail_pass' column to create the 'Pass/Fail' column (1=fail, 0=pass)

# Keep only target and features
columns_to_drop = [
    'DevID',
    'WAFER_NO',
    'Pass/Fail_pass'
]
processed_dataset.drop(columns=columns_to_drop, inplace=True)

In [21]:
# Convert categorical columns to dummy columns

processed_dataset = pd.get_dummies(processed_dataset, drop_first=True) # One-hot encode categorical columns (drop the first category)

# Convert all columns to numeric type
processed_dataset = processed_dataset.apply(pd.to_numeric)
processed_dataset.fillna(processed_dataset.mean(), inplace=True) # Fill missing values with the mean of the respective column

# Clean up column names
processed_dataset.columns = (
    processed_dataset.columns.str.replace("[", "_", regex=False) # Replace '[' with '_'
    .str.replace("]", "_", regex=False) # Replace ']' with '_'
    .str.replace("<", "_", regex=False) # Replace '<' with '_'
    .str.replace(">", "_", regex=False) # Replace '>' with '_'
)

# Reorder columns to place the target column, 'Pass/Fail', last
reorder_cols = [c for c in processed_dataset.columns if c not in ["Pass/Fail"]] # Select all columns except 'Pass/Fail'
processed_dataset = processed_dataset[reorder_cols + ["Pass/Fail"]] # Reorder columns by adding 'Pass/Fail' at the end

print("      Preprocessing complete!\n")
# processed_dataset

      Preprocessing complete!



# create_train_and_test_data


In [22]:
# Split data into 8:2
split_parameter = copy.deepcopy(split_parameter_default)
train_data, test_data, split_parameter_info = create_train_test_data(processed_dataset, split_parameter)



##############################################################################################################################
# 3) Create Train/Test Split 
##############################################################################################################################

      Creating training and test datasets...
    - Not applying filtering before splitting.
    - Not applying Feature Generation.

    - Training data class distribution before splitting: {0.0: 3546, 1.0: 71}
    - No sampling applied


# select_feature


### Calculating feature importance: variance, linear/nonlinear correlation coefficient, model importance (sfm:select from model)

In [23]:
# Set all feature selection

feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)

feature_selector_params_licor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_licor["filter_methods"]["apply_target_linear_corr_filter"] = True
feature_selector_params_licor["filter_methods"]["target_linear_corr_threshold"] = 0.00
feature_selection_info_licor = select_feature(train_data, feature_selector_params_licor)

feature_selector_params_xicor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_xicor["filter_methods"]["apply_target_xicor_filter"] = True
feature_selector_params_xicor["filter_methods"]["target_xicor_threshold"] = 0.00
feature_selection_info_xicor = select_feature(train_data, feature_selector_params_xicor)

feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "0*median"
feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

feature_selection_infos = {
     "var" : feature_selection_info_var,
     "licor" : feature_selection_info_licor,
     "xicor" : feature_selection_info_xicor,
     "model" : feature_selection_info_sfm
}
for fileter_name, feature_selection_info in feature_selection_infos.items():
    print("# of feature:", feature_selection_info["final_feature_count"], ",  filter: ", feature_selection_info["feature_selector_name"], fileter_name)

# # Feature selection results by method: dic
# feature_selection_info

# # Feature selection results summary data by method: df
# features_values_dfs


--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - Number of features remaining after variance filtering: 1401

Feature selection results saved to 'result/feature_selection/jsons\feature_selection_info_250904_113312_f79537d2.json' file.

- Final feature count: 1401

--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - Number of features remaining after target linear correlation filtering: 1650

Feature selection results saved to 'result/feature_selection/jsons\feature_selection_info_250904_113312_0e5393fb.json' file.

- Final feature count: 1650

--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - 타겟 Xi Cor 필터링 후 남은 피처 수: 1650

Feature selection results saved to 'result/feature_selection/jsons\feature_selection_info_250904_113313_6060c7d1.json' file.

- Final feature count: 1650

--- Feature Selector: SFM ---
--- SFM Selector completed ---
Number of remaining features: 1650

Feature selection res

In [24]:
# Feature importance information summary: features_values_dfs

features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    features_values_dfs = pd.merge(
        features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


# features_values_dfs

In [25]:
# Check prediction performance: Select all features

# Apply the model based on the user specified threshold and summarize the results
# feature select test - submit and summary result

usr_pl_fs_test_result_ftpn_df = pd.DataFrame()
usr_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test_rf(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    usr_pl_fs_test_result_ftpn_df = pd.concat([usr_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    usr_pl_fs_test_result_features_values_dfs = pd.merge(
        usr_pl_fs_test_result_features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


# # Model performance summary of the feature selection set - based on user feature selection threshold
# usr_pl_fs_test_result_ftpn_df

# # Model performance summary of the feature selection set + applied columns from user feature selection threshold (for reference)
# usr_pl_fs_test_result_features_values_dfs

      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
    Best F2 (class=1) score (CV): 0.2180

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.7980 with F2 score: 0.6199
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
    Best F2 (class=1) score (CV): 0.2151

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.7778 with F2 score: 0.6095
     Calculation of the ROC curve...
     Calculation done
     Scoring...
   

### Optimal Feature Selection Search

##### The optimal selection method uses the XGBClassifier model and GridSearchCV for searching for optimal feature selection threshold values


In [26]:
# Utilizing feature importance information
feature_importance_df = features_values_dfs.copy()

In [27]:
# Feature selection prediction performance check: Use only training data

target_df = train_data.iloc[:, -1]
features_df = train_data.iloc[:, :-1]
X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, test_size=0.2, random_state=42)


# run_optimization_for_feature_importance : a function to select features based on a specific importance column and find the optimal model
def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    A function to select features based on a specific importance column and find the optimal model.
    
    Parameters:
    - train_data (pd.DataFrame): Training data
    - target_data (pd.Series): Target data
    - feature_importance_df (pd.DataFrame): DataFrame containing feature importance information
    - importance_column (str): The column name to determine the importance rank
    - k_percentiles (list): List of candidate percentiles for the number of features to select (e.g., [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: Optimized feature importance information
    - pd.DataFrame: Model performance summary information
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # Select the top K features based on the values in the importance column
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # Convert percentiles to the actual number of features
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ Variable to store the optimal percentile value
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # Prepare the dataset with only the optimal features
        X_train_filtered = train_data[top_k_features]
        
        # Model training pipeline (direct feature selection instead of SelectKBest)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # Evaluate the performance for the current K
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ Update the best percentile
            best_k_percentile = k_percentiles[i]

    # Generate feature importance and performance information for the optimal model
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # Assign importance scores only to the selected features
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # Final performance evaluation with the test data
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ Add the optimal percentile column
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary


In [28]:
# Find the optimal feature selection by adjusting the feature selection threshold (k_percentiles): Result summary file -> final_feature_performance_summary_df

# Repeat optimization for each importance column and accumulate results
# Save the importance column names from the 2nd column onwards, excluding the feature name column
importance_cols = feature_importance_df.columns[1:].tolist()

# ⭐️ Change to a list of percentile candidates: defined by the user
k_percentiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- Running optimization based on {col} column ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_feature_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. Print results and save to CSV
# print("\n--- Final accumulated feature importance information ---")
# print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

# print("\n--- Final accumulated model performance summary ---")
# print(final_feature_performance_summary_df)
final_feature_performance_summary_df.to_csv('final_feature_performance_summary.csv', index=False)

# # Automatic (optimal) feature selection result data: df
# final_feature_info_df

# # Automatic (optimal) feature selection result data summary: df
# final_feature_performance_summary_df


--- Running optimization based on FeatureFilter_variance column ---

--- Running optimization based on FeatureFilter_target_linear_correlation column ---

--- Running optimization based on FeatureFilter_target_xicor_correlation column ---

--- Running optimization based on SFM_importances column ---


##### Optimal feature selection result (best_k_percentile, final_feature_count): final_feature_performance_summary_df

In [29]:
# Automatic (optimal) feature selection result data summary: df
final_feature_performance_summary_df


,fn,fp,tn,tp,feature_selector_name,initial_feature_count,final_feature_count,f2_score,importance_column,best_k_percentile
0,14,7,703,0,FeatureFilter_variance,1650,1320,0.0,FeatureFilter_variance,0.80
1,14,7,703,0,FeatureFilter_target_linear_correlation,1650,165,0.0,FeatureFilter_target_linear_correlation,0.10
2,14,7,703,0,FeatureFilter_target_xicor_correlation,1650,1485,0.0,FeatureFilter_target_xicor_correlation,0.90
3,14,6,704,0,SFM_importances,1650,82,0.0,SFM_importances,0.05


# Verifying the predictive performance of optimal feature selection: Applying modeling

### Applying modeling : rf_cv

In [74]:
# Prepare modeling parameters: Final (optimal) feature set -> feature_selection_results(dic)

# Prepare performance check pipeline input: feature_selection_results
feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [75]:
# Preparing the modeling pipeline: rf_cv algorithm
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_final_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    # trained_model, feature_importance, train_parameters_info = train_model_xgboost_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


In [76]:
# Running the modeling pipeline and summarizing the results

# Optimal feature set performance check
# feature select test - submit and summary result

trained_model_set = {}
feature_importance_set = pd.DataFrame()
forecast_dataset_set = pd.DataFrame()
train_dataset_proba_set = pd.DataFrame()
best_threshold_set = pd.DataFrame()
roc_data_set = pd.DataFrame()
auc_score_set = pd.DataFrame()
train_dataset_metrics_set = pd.DataFrame()
metrics_set = pd.DataFrame()
results_set = pd.DataFrame()

opt_pl_fs_test_result_ftpn_df = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for feature_selector_idx, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_final_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            # train_parameters_list_default["xgboost_cv"],
            test_data
            )

    trained_model_set[feature_selector_idx] = trained_model

    feature_importance["feature_selector_idx"] = feature_selector_idx
    feature_importance_set = pd.concat([feature_importance_set, feature_importance], ignore_index=True)

    forecast_dataset_df = pd.DataFrame({
        "feature_selector_idx": [feature_selector_idx] * len(forecast_dataset),
        "forecast_prob": forecast_dataset
    })
    forecast_dataset_set = pd.concat([forecast_dataset_set, forecast_dataset_df], ignore_index=True)
    
    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    best_threshold_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "best_threshold": [best_threshold]
    })
    best_threshold_set = pd.concat([best_threshold_set, best_threshold_df], ignore_index=True)

    roc_data["feature_selector_idx"] = feature_selector_idx
    roc_data_set = pd.concat([roc_data_set, roc_data], ignore_index=True)

    auc_score_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "auc_score": [auc_score]
    })
    auc_score_set = pd.concat([auc_score_set, auc_score_df], ignore_index=True)

    train_dataset_metrics["feature_selector_idx"] = feature_selector_idx
    train_dataset_metrics_set = pd.concat([train_dataset_metrics_set, train_dataset_metrics], ignore_index=True)

    metrics_dict = {
        "f1_score": metrics["f1_score"],
        "recall": metrics["recall"],
        "precision": metrics["precision"],
        "accuracy": metrics["accuracy"],
        "auc_score": metrics["auc_score"],
        "tp": metrics["dict_ftpn"]["tp"],
        "tn": metrics["dict_ftpn"]["tn"],
        "fp": metrics["dict_ftpn"]["fp"],
        "fn": metrics["dict_ftpn"]["fn"],
        "number_of_predictions": metrics["number_of_predictions"],
        "number_of_good_predictions": metrics["number_of_good_predictions"],
        "number_of_false_predictions": metrics["number_of_false_predictions"],
        "feature_selector_idx": feature_selector_idx  # 현재 필터 이름 추가
    }
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_set = pd.concat([metrics_set, metrics_df], ignore_index=True)

    results["feature_selector_idx"] = feature_selector_idx
    results_set = pd.concat([results_set, results], ignore_index=True)

    dict_ftpn = metrics["dict_ftpn"]
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df = pd.concat([opt_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })
    opt_pl_fs_test_result_features_values_dfs = pd.merge(
        opt_pl_fs_test_result_features_values_dfs,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column


      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2325

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.6062
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.1980

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.7980 with F2 score: 0.5312
     Calculation of the ROC curve...
     Calculation done
     Scoring...
    

In [77]:
# Modeling pipeline execution result output: data -> csv

save_path = "result/feature_selection/eval/rf_cv"
os.makedirs(save_path, exist_ok=True)

dict_vars = [
    (trained_model_set, "trained_model_set")
]

df_vars = [
    (feature_importance_set, "feature_importance_set"),
    (forecast_dataset_set, "forecast_dataset_set"),
    (train_dataset_proba_set, "train_dataset_proba_set"),
    (best_threshold_set, "best_threshold_set"),
    (roc_data_set, "roc_data_set"),
    (auc_score_set, "auc_score_set"),
    (train_dataset_metrics_set, "train_dataset_metrics_set"),
    (metrics_set, "metrics_set"),
    (results_set, "results_set"),
    (opt_pl_fs_test_result_ftpn_df, "opt_pl_fs_test_result_ftpn_df"),
    (opt_pl_fs_test_result_features_values_dfs, "opt_pl_fs_test_result_features_values_dfs")
]

for var, name in dict_vars:
    file_path = os.path.join(save_path, f"{name}.pickle")
    with open(file_path, "wb") as f:
        pickle.dump(var, f)
    print(f"Saved {name} to {file_path}")

for var, name in df_vars:
    file_path = os.path.join(save_path, f"{name}.csv")
    var.to_csv(file_path, index=False)
    print(f"Saved {name} to {file_path}")
    

# opt_pl_fs_test_result_ftpn_df
opt_pl_fs_test_result_features_values_dfs

Saved trained_model_set to result/feature_selection/eval/rf_cv\trained_model_set.pickle
Saved feature_importance_set to result/feature_selection/eval/rf_cv\feature_importance_set.csv
Saved forecast_dataset_set to result/feature_selection/eval/rf_cv\forecast_dataset_set.csv
Saved train_dataset_proba_set to result/feature_selection/eval/rf_cv\train_dataset_proba_set.csv
Saved best_threshold_set to result/feature_selection/eval/rf_cv\best_threshold_set.csv
Saved roc_data_set to result/feature_selection/eval/rf_cv\roc_data_set.csv
Saved auc_score_set to result/feature_selection/eval/rf_cv\auc_score_set.csv
Saved train_dataset_metrics_set to result/feature_selection/eval/rf_cv\train_dataset_metrics_set.csv
Saved metrics_set to result/feature_selection/eval/rf_cv\metrics_set.csv
Saved results_set to result/feature_selection/eval/rf_cv\results_set.csv
Saved opt_pl_fs_test_result_ftpn_df to result/feature_selection/eval/rf_cv\opt_pl_fs_test_result_ftpn_df.csv
Saved opt_pl_fs_test_result_featur

,feature_name,Importance(model)_FeatureFilter_variance,Importance(model)_FeatureFilter_target_linear_correlation,Importance(model)_FeatureFilter_target_xicor_correlation,Importance(model)_SFM_importances
0,WF of 1103959_69_1133529_cp1,0.0,NaN,NaN,NaN
1,ROW of 1103959_69_1133529_cp1,0.0,NaN,0.000677,NaN
2,COL of 1103959_69_1133529_cp1,0.0,NaN,0.011295,NaN
3,S of 1103959_69_1133529_cp1,NaN,NaN,0.000000,NaN
4,HH_CONT_OUT of 1103959_69_1133529_cp1,0.0,NaN,0.000000,NaN
...,...,...,...,...,...
1646,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,NaN,NaN,0.000000,NaN
1647,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,NaN,NaN,0.000000,NaN
1648,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,NaN,NaN,0.000000,NaN
1649,band gap dpat_ok for band gap,NaN,NaN,0.000000,NaN


In [78]:
# Display modeling pipeline execution results: Chart (best feature set only)

df = opt_pl_fs_test_result_ftpn_df

# Sort the DataFrame according to the specified conditions
# 'fn' ascending, 'fp' ascending, 'final_feature_count' ascending
sorted_df = df.sort_values(by=['fn', 'fp', 'final_feature_count'], ascending=[True, True, True])

# Store the 'feature_selector_idx' value of the highest priority data
best_idx = sorted_df.iloc[0]['feature_selector_idx']

# Print results
print(f"Sorted DataFrame:\n{sorted_df}\n")
print(f"Best feature_selector_idx: {best_idx}")

Sorted DataFrame:
   fn   fp   tn  tp  feature_selector_idx  \
1   3  201  686  15                     1   
0   4  174  713  14                     0   
2   6  165  722  12                     2   
3   6  211  676  12                     3   

                     feature_selector_name  initial_feature_count  \
1  FeatureFilter_target_linear_correlation                   1650   
0                   FeatureFilter_variance                   1650   
2   FeatureFilter_target_xicor_correlation                   1650   
3                          SFM_importances                   1650   

   final_feature_count  final_feature_selector_f2score  \
1                  165                             0.0   
0                 1320                             0.0   
2                 1485                             0.0   
3                   82                             0.0   

   final_feature_selector_best_k_percentile  
1                                      0.10  
0                          

In [79]:
#--- ROC chart

try:
    roc_data_set_df = pd.read_csv(os.path.join(save_path, 'roc_data_set.csv'))
    print("DataFrame 'roc_data_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(roc_data_set_df.head())
except FileNotFoundError:
    print("Error: The file was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'roc_data_set_df' has been successfully loaded.


In [ ]:
# 1. Filter data where 'feature_selector_idx' is best_idx
filtered_df = roc_data_set_df[roc_data_set_df['feature_selector_idx'] == best_idx].copy()

# 2. Add (0,0) and (1,1) points to complete the curve
filtered_df.loc[-1] = {'False positive rate': 0.0, 'True positive rate': 0.0, 'feature_selector_idx': 0}
filtered_df.loc[len(filtered_df)] = {'False positive rate': 1.0, 'True positive rate': 1.0, 'feature_selector_idx': 0}
filtered_df.sort_values(by='False positive rate', inplace=True)

# 3. Calculate AUC value
roc_auc = auc(filtered_df['False positive rate'], filtered_df['True positive rate'])
roc_auc = round(roc_auc, 4)

# 4. Create chart with Plotly Express
fig = px.area(
    filtered_df,
    x='False positive rate',
    y='True positive rate',
    labels={'False positive rate': 'False Positive Rate', 'True positive rate': 'True Positive Rate'}
)

# Add random classifier (dotted line)
fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='black'),
    name='Random Classifier'
)

# 5. Set layout and AUC annotation
fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1], constrain='domain'),
    yaxis=dict(range=[0, 1], scaleanchor='x', scaleratio=1),
    showlegend=False,
    width=500,
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5,
    annotations=[
        dict(
            xref='paper',
            yref='paper',
            x=0,
            y=1.1,
            text=f'ROC Curve (AUC={roc_auc})',
            showarrow=False,
            font=dict(size=14, color='black'),
            align='left'
        )
    ]
)

# 6. Save image to the specified folder
file_path = os.path.join(save_path, 'ROC_Curve.png')

# Create folder if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save as image file
fig.write_image(file_path)

# Display chart in VS Code
fig.show()

print(f"ROC chart has been saved to {file_path}.")

ROC chart has been saved to result/feature_selection/eval/rf_cv\ROC_Curve.png.


In [81]:
#--- Confusion matrix chart

try:
    results_set_df = pd.read_csv(os.path.join(save_path, 'results_set.csv'))
    print("DataFrame 'results_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(results_set_df.head())
except FileNotFoundError:
    print("Error: The file 'data/result/select_feature/results_set.csv' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'results_set_df' has been successfully loaded.


In [82]:
# 1. Filter the DataFrame for 'feature_selector_idx' = best_idx
filtered_df = results_set_df[results_set_df['feature_selector_idx'] == best_idx].copy()

# A more robust way to handle this is to use both 'Historical' and 'Forecast' as separate data.
# However, since the provided image only shows one classification result column, we'll
# deduce the true and predicted values from it.

# Let's use the columns as they should be, based on standard ML practice
y_actual = filtered_df['Historical']
y_predicted = filtered_df['Forecast']

# Re-mapping based on actual and predicted outcomes
# Target is '1' (Failure).
# Actuals: 1 for Fail, 0 for Pass
# Predicted: 1 for Fail, 0 for Pass
y_true_binary = y_actual.apply(lambda x: 1 if x == 1 else 0)
y_pred_binary = y_predicted.apply(lambda x: 1 if x == 1 else 0)

# 3. Calculate the confusion matrix
# The `labels` argument ensures the order of the matrix is [Positive, Negative]
# The provided image shows Predicted Pass (0) before Predicted Fail (1) and
# True Pass (0) before True Fail (1). Let's adjust the labels accordingly.
cm = confusion_matrix(y_true_binary, y_pred_binary, labels=[0, 1])

# 4. Create a DataFrame for the heatmap with correct labels
# The provided image shows 'True label 0' (Pass) at the top and 'True label 1' (Fail) at the bottom.
# It also shows 'Predicted label 0' (Pass) on the left and 'Predicted label 1' (Fail) on the right.
# We need to reflect this order in our labels.
cm_df = pd.DataFrame(
    cm,
    index=['True (Actual Pass)', 'True (Actual Fail)'],
    columns=['Predicted Pass', 'Predicted Fail']
)

# 5. Plot the confusion matrix using Plotly Express
fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='blues')

# Update layout for a better visualization
fig.update_layout(
    title='Train Set', # Title is now 'Train Set'
    xaxis_title='Predicted label',
    yaxis_title='True label',
    width=500,
    height=500,
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

# Save the figure to a file
file_path = os.path.join(save_path, 'confusion_matrix.png')

os.makedirs(os.path.dirname(file_path), exist_ok=True)
fig.write_image(file_path)

fig.show()

print(f"Corrected confusion matrix chart has been generated and saved to {file_path}.")

Corrected confusion matrix chart has been generated and saved to result/feature_selection/eval/rf_cv\confusion_matrix.png.


In [83]:
try:
    feature_importance_set_df = pd.read_csv(os.path.join(save_path, 'feature_importance_set.csv'))
    print("DataFrame 'feature_importance_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(feature_importance_set_df.head())
except FileNotFoundError:
    print("Error: The file  was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'feature_importance_set_df' has been successfully loaded.


In [84]:
# 1. Filter data where 'feature_selector_idx' is best_idx
filtered_df = feature_importance_set_df[feature_importance_set_df['feature_selector_idx'] == best_idx].copy()

# 2. Sort by 'Importance_abs' in descending order
sorted_df = filtered_df.sort_values(by='Importance_abs', ascending=False)

# 3. Select the top 20 features
top_20_df = sorted_df.head(20)

# 4. Create a horizontal bar chart
fig = px.bar(
    top_20_df,
    x='Importance',
    y='Features',
    orientation='h',
    title='Feature Importance',
    labels={'Importance': 'Importance', 'Features': 'Feature'}
)

# 5. Update layout (set horizontal size to 1200)
# This setting applies to both Jupyter notebook output and file saving.
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Importance',
    yaxis_title='Feature',
    width=1500,  # Ensure this value matches your desired notebook display width
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5
)

# Save as an image file

file_path = os.path.join(save_path, 'feature_importance_wide.png')
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the image file, explicitly setting dimensions
fig.write_image(
    file_path,
    width=1500,
    height=500
)

# Display chart
fig.show()

print(f"The feature importance chart with expanded horizontal size has been created and saved to {file_path}.")

The feature importance chart with expanded horizontal size has been created and saved to result/feature_selection/eval/rf_cv\feature_importance_wide.png.


### Applying modeling : xgboost_cv

In [85]:
# Prepare modeling parameters: Final (optimal) feature set -> feature_selection_results(dic)

# Prepare performance check pipeline input: feature_selection_results
feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [86]:
# Preparing the modeling pipeline: xgboost_cv algorithm
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_final_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    # trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    trained_model, feature_importance, train_parameters_info = train_model_xgboost_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


In [87]:
# Running the modeling pipeline and summarizing the results

# Optimal feature set performance check
# feature select test - submit and summary result

trained_model_set = {}
feature_importance_set = pd.DataFrame()
forecast_dataset_set = pd.DataFrame()
train_dataset_proba_set = pd.DataFrame()
best_threshold_set = pd.DataFrame()
roc_data_set = pd.DataFrame()
auc_score_set = pd.DataFrame()
train_dataset_metrics_set = pd.DataFrame()
metrics_set = pd.DataFrame()
results_set = pd.DataFrame()

opt_pl_fs_test_result_ftpn_df = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for feature_selector_idx, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_final_test(
            train_data,
            feature_selection_info,
            # train_parameters_list_default["rf_cv"],
            train_parameters_list_default["xgboost_cv"],
            test_data
            )

    trained_model_set[feature_selector_idx] = trained_model

    feature_importance["feature_selector_idx"] = feature_selector_idx
    feature_importance_set = pd.concat([feature_importance_set, feature_importance], ignore_index=True)

    forecast_dataset_df = pd.DataFrame({
        "feature_selector_idx": [feature_selector_idx] * len(forecast_dataset),
        "forecast_prob": forecast_dataset
    })
    forecast_dataset_set = pd.concat([forecast_dataset_set, forecast_dataset_df], ignore_index=True)
    
    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    best_threshold_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "best_threshold": [best_threshold]
    })
    best_threshold_set = pd.concat([best_threshold_set, best_threshold_df], ignore_index=True)

    roc_data["feature_selector_idx"] = feature_selector_idx
    roc_data_set = pd.concat([roc_data_set, roc_data], ignore_index=True)

    auc_score_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "auc_score": [auc_score]
    })
    auc_score_set = pd.concat([auc_score_set, auc_score_df], ignore_index=True)

    train_dataset_metrics["feature_selector_idx"] = feature_selector_idx
    train_dataset_metrics_set = pd.concat([train_dataset_metrics_set, train_dataset_metrics], ignore_index=True)

    metrics_dict = {
        "f1_score": metrics["f1_score"],
        "recall": metrics["recall"],
        "precision": metrics["precision"],
        "accuracy": metrics["accuracy"],
        "auc_score": metrics["auc_score"],
        "tp": metrics["dict_ftpn"]["tp"],
        "tn": metrics["dict_ftpn"]["tn"],
        "fp": metrics["dict_ftpn"]["fp"],
        "fn": metrics["dict_ftpn"]["fn"],
        "number_of_predictions": metrics["number_of_predictions"],
        "number_of_good_predictions": metrics["number_of_good_predictions"],
        "number_of_false_predictions": metrics["number_of_false_predictions"],
        "feature_selector_idx": feature_selector_idx  # 현재 필터 이름 추가
    }
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_set = pd.concat([metrics_set, metrics_df], ignore_index=True)

    results["feature_selector_idx"] = feature_selector_idx
    results_set = pd.concat([results_set, results], ignore_index=True)

    dict_ftpn = metrics["dict_ftpn"]
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df = pd.concat([opt_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })
    opt_pl_fs_test_result_features_values_dfs = pd.merge(
        opt_pl_fs_test_result_features_values_dfs,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column


     Training the XGBoost model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

     Best parameters found: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100}
     Best F2 (class=1) score: 0.2367

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.8182 with F2 score: 0.8895
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
     Training the XGBoost model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

     Best parameters found: {'learning_rate': 0.2, 'max_depth': 2, 'n_estimators': 50}
     Best F2 (class=1) score: 0.2603

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.8384 with F2 score: 0.9285
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Cr

In [88]:
# Modeling pipeline execution result output: data -> csv

save_path = "result/feature_selection/eval/xgboost_cv"
os.makedirs(save_path, exist_ok=True)

dict_vars = [
    (trained_model_set, "trained_model_set")
]

df_vars = [
    (feature_importance_set, "feature_importance_set"),
    (forecast_dataset_set, "forecast_dataset_set"),
    (train_dataset_proba_set, "train_dataset_proba_set"),
    (best_threshold_set, "best_threshold_set"),
    (roc_data_set, "roc_data_set"),
    (auc_score_set, "auc_score_set"),
    (train_dataset_metrics_set, "train_dataset_metrics_set"),
    (metrics_set, "metrics_set"),
    (results_set, "results_set"),
    (opt_pl_fs_test_result_ftpn_df, "opt_pl_fs_test_result_ftpn_df"),
    (opt_pl_fs_test_result_features_values_dfs, "opt_pl_fs_test_result_features_values_dfs")
]

for var, name in dict_vars:
    file_path = os.path.join(save_path, f"{name}.pickle")
    with open(file_path, "wb") as f:
        pickle.dump(var, f)
    print(f"Saved {name} to {file_path}")

for var, name in df_vars:
    file_path = os.path.join(save_path, f"{name}.csv")
    var.to_csv(file_path, index=False)
    print(f"Saved {name} to {file_path}")
    

# opt_pl_fs_test_result_ftpn_df
opt_pl_fs_test_result_features_values_dfs

Saved trained_model_set to result/feature_selection/eval/xgboost_cv\trained_model_set.pickle
Saved feature_importance_set to result/feature_selection/eval/xgboost_cv\feature_importance_set.csv
Saved forecast_dataset_set to result/feature_selection/eval/xgboost_cv\forecast_dataset_set.csv
Saved train_dataset_proba_set to result/feature_selection/eval/xgboost_cv\train_dataset_proba_set.csv
Saved best_threshold_set to result/feature_selection/eval/xgboost_cv\best_threshold_set.csv
Saved roc_data_set to result/feature_selection/eval/xgboost_cv\roc_data_set.csv
Saved auc_score_set to result/feature_selection/eval/xgboost_cv\auc_score_set.csv
Saved train_dataset_metrics_set to result/feature_selection/eval/xgboost_cv\train_dataset_metrics_set.csv
Saved metrics_set to result/feature_selection/eval/xgboost_cv\metrics_set.csv
Saved results_set to result/feature_selection/eval/xgboost_cv\results_set.csv
Saved opt_pl_fs_test_result_ftpn_df to result/feature_selection/eval/xgboost_cv\opt_pl_fs_tes

,feature_name,Importance(model)_FeatureFilter_variance,Importance(model)_FeatureFilter_target_linear_correlation,Importance(model)_FeatureFilter_target_xicor_correlation,Importance(model)_SFM_importances
0,WF of 1103959_69_1133529_cp1,0.000000,NaN,NaN,NaN
1,ROW of 1103959_69_1133529_cp1,0.000000,NaN,0.000000,NaN
2,COL of 1103959_69_1133529_cp1,0.044775,NaN,0.048186,NaN
3,S of 1103959_69_1133529_cp1,NaN,NaN,0.000000,NaN
4,HH_CONT_OUT of 1103959_69_1133529_cp1,0.000505,NaN,0.000000,NaN
...,...,...,...,...,...
1646,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,NaN,NaN,0.000000,NaN
1647,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,NaN,NaN,0.000000,NaN
1648,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,NaN,NaN,0.000000,NaN
1649,band gap dpat_ok for band gap,NaN,NaN,0.000000,NaN


In [89]:
# Display modeling pipeline execution results: Chart (best feature set only)

df = opt_pl_fs_test_result_ftpn_df

# Sort the DataFrame according to the specified conditions
# 'fn' ascending, 'fp' ascending, 'final_feature_count' ascending
sorted_df = df.sort_values(by=['fn', 'fp', 'final_feature_count'], ascending=[True, True, True])

# Store the 'feature_selector_idx' value of the highest priority data
best_idx = sorted_df.iloc[0]['feature_selector_idx']

# Print results
print(f"Sorted DataFrame:\n{sorted_df}\n")
print(f"Best feature_selector_idx: {best_idx}")

Sorted DataFrame:
   fn  fp   tn  tp  feature_selector_idx  \
0  15  34  853   3                     0   
1  16  21  866   2                     1   
2  16  28  859   2                     2   
3  16  52  835   2                     3   

                     feature_selector_name  initial_feature_count  \
0                   FeatureFilter_variance                   1650   
1  FeatureFilter_target_linear_correlation                   1650   
2   FeatureFilter_target_xicor_correlation                   1650   
3                          SFM_importances                   1650   

   final_feature_count  final_feature_selector_f2score  \
0                 1320                             0.0   
1                  165                             0.0   
2                 1485                             0.0   
3                   82                             0.0   

   final_feature_selector_best_k_percentile  
0                                      0.80  
1                               

In [90]:
#--- ROC chart

try:
    roc_data_set_df = pd.read_csv(os.path.join(save_path, 'roc_data_set.csv'))
    print("DataFrame 'roc_data_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(roc_data_set_df.head())
except FileNotFoundError:
    print("Error: The file was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'roc_data_set_df' has been successfully loaded.


In [91]:
# 1. Filter data where 'feature_selector_idx' is best_idx
filtered_df = roc_data_set_df[roc_data_set_df['feature_selector_idx'] == best_idx].copy()

# 2. Add (0,0) and (1,1) points to complete the curve
filtered_df.loc[-1] = {'False positive rate': 0.0, 'True positive rate': 0.0, 'feature_selector_idx': 0}
filtered_df.loc[len(filtered_df)] = {'False positive rate': 1.0, 'True positive rate': 1.0, 'feature_selector_idx': 0}
filtered_df.sort_values(by='False positive rate', inplace=True)

# 3. Calculate AUC value
roc_auc = auc(filtered_df['False positive rate'], filtered_df['True positive rate'])
roc_auc = round(roc_auc, 4)

# 4. Create chart with Plotly Express
fig = px.area(
    filtered_df,
    x='False positive rate',
    y='True positive rate',
    labels={'False positive rate': 'False Positive Rate', 'True positive rate': 'True Positive Rate'}
)

# Add random classifier (dotted line)
fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='black'),
    name='Random Classifier'
)

# 5. Set layout and AUC annotation
fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1], constrain='domain'),
    yaxis=dict(range=[0, 1], scaleanchor='x', scaleratio=1),
    showlegend=False,
    width=500,
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5,
    annotations=[
        dict(
            xref='paper',
            yref='paper',
            x=0,
            y=1.1,
            text=f'ROC Curve (AUC={roc_auc})',
            showarrow=False,
            font=dict(size=14, color='black'),
            align='left'
        )
    ]
)

# 6. Save image to the specified folder
file_path = os.path.join(save_path, 'ROC_Curve.png')

# Create folder if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save as image file
fig.write_image(file_path)

# Display chart in VS Code
fig.show()

print(f"ROC chart has been saved to {file_path}.")

ROC chart has been saved to result/feature_selection/eval/xgboost_cv\ROC_Curve.png.


In [92]:
#--- Confusion matrix chart

try:
    results_set_df = pd.read_csv(os.path.join(save_path, 'results_set.csv'))
    print("DataFrame 'results_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(results_set_df.head())
except FileNotFoundError:
    print("Error: The file 'data/result/select_feature/results_set.csv' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'results_set_df' has been successfully loaded.


In [93]:
# 1. Filter the DataFrame for 'feature_selector_idx' = best_idx
filtered_df = results_set_df[results_set_df['feature_selector_idx'] == best_idx].copy()

# A more robust way to handle this is to use both 'Historical' and 'Forecast' as separate data.
# However, since the provided image only shows one classification result column, we'll
# deduce the true and predicted values from it.

# Let's use the columns as they should be, based on standard ML practice
y_actual = filtered_df['Historical']
y_predicted = filtered_df['Forecast']

# Re-mapping based on actual and predicted outcomes
# Target is '1' (Failure).
# Actuals: 1 for Fail, 0 for Pass
# Predicted: 1 for Fail, 0 for Pass
y_true_binary = y_actual.apply(lambda x: 1 if x == 1 else 0)
y_pred_binary = y_predicted.apply(lambda x: 1 if x == 1 else 0)

# 3. Calculate the confusion matrix
# The `labels` argument ensures the order of the matrix is [Positive, Negative]
# The provided image shows Predicted Pass (0) before Predicted Fail (1) and
# True Pass (0) before True Fail (1). Let's adjust the labels accordingly.
cm = confusion_matrix(y_true_binary, y_pred_binary, labels=[0, 1])

# 4. Create a DataFrame for the heatmap with correct labels
# The provided image shows 'True label 0' (Pass) at the top and 'True label 1' (Fail) at the bottom.
# It also shows 'Predicted label 0' (Pass) on the left and 'Predicted label 1' (Fail) on the right.
# We need to reflect this order in our labels.
cm_df = pd.DataFrame(
    cm,
    index=['True (Actual Pass)', 'True (Actual Fail)'],
    columns=['Predicted Pass', 'Predicted Fail']
)

# 5. Plot the confusion matrix using Plotly Express
fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='blues')

# Update layout for a better visualization
fig.update_layout(
    title='Train Set', # Title is now 'Train Set'
    xaxis_title='Predicted label',
    yaxis_title='True label',
    width=500,
    height=500,
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

# Save the figure to a file
file_path = os.path.join(save_path, 'confusion_matrix.png')

os.makedirs(os.path.dirname(file_path), exist_ok=True)
fig.write_image(file_path)

fig.show()

print(f"Corrected confusion matrix chart has been generated and saved to {file_path}.")

Corrected confusion matrix chart has been generated and saved to result/feature_selection/eval/xgboost_cv\confusion_matrix.png.


In [94]:
try:
    feature_importance_set_df = pd.read_csv(os.path.join(save_path, 'feature_importance_set.csv'))
    print("DataFrame 'feature_importance_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(feature_importance_set_df.head())
except FileNotFoundError:
    print("Error: The file  was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'feature_importance_set_df' has been successfully loaded.


In [95]:
# 1. Filter data where 'feature_selector_idx' is best_idx
filtered_df = feature_importance_set_df[feature_importance_set_df['feature_selector_idx'] == best_idx].copy()

# 2. Sort by 'Importance_abs' in descending order
sorted_df = filtered_df.sort_values(by='Importance_abs', ascending=False)

# 3. Select the top 20 features
top_20_df = sorted_df.head(20)

# 4. Create a horizontal bar chart
fig = px.bar(
    top_20_df,
    x='Importance',
    y='Features',
    orientation='h',
    title='Feature Importance',
    labels={'Importance': 'Importance', 'Features': 'Feature'}
)

# 5. Update layout (set horizontal size to 1200)
# This setting applies to both Jupyter notebook output and file saving.
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Importance',
    yaxis_title='Feature',
    width=1500,  # Ensure this value matches your desired notebook display width
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5
)

# Save as an image file

file_path = os.path.join(save_path, 'feature_importance_wide.png')
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save the image file, explicitly setting dimensions
fig.write_image(
    file_path,
    width=1500,
    height=500
)

# Display chart
fig.show()

print(f"The feature importance chart with expanded horizontal size has been created and saved to {file_path}.")

The feature importance chart with expanded horizontal size has been created and saved to result/feature_selection/eval/xgboost_cv\feature_importance_wide.png.
